# Day 4: Session 4C - The Derived-Column Pattern

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/4c_transforming_data.html)

Date: 09/03/2026

In [1]:
import pandas as pd
import numpy as np

url = 'https://eds-217-essential-python.github.io/data/marine_microplastics.csv'
plastics = pd.read_csv(url)

m3 = plastics[plastics['Unit'] == 'pieces/m3'].copy()
m3.shape

(10178, 22)

### The pattern

In [ ]:
m3['pieces_per_liter'] = m3['Measurement'] / 1000

# made a new column called pieces_per_liter

# this is the derived-column pattern. The general form is:
# df['new_column'] = expression

In [4]:
m3[['Measurement', 'Unit', 'pieces_per_liter']].head()

,Measurement,Unit,pieces_per_liter
0,0.020000,pieces/m3,0.000020
1,0.008000,pieces/m3,0.000008
2,0.019886,pieces/m3,0.000020
3,0.018000,pieces/m3,0.000018
4,0.000000,pieces/m3,0.000000


In [ ]:
m3['pieces_per_liter'].describe()

count    10178.000000
mean         0.219409
std          2.599555
min          0.000000
25%          0.000000
50%          0.000007
75%          0.000050
max        110.480000
Name: pieces_per_liter, dtype: float64

In [7]:
m3.shape

(10178, 23)

In [11]:
m3['latitude_radians'] = m3['Latitude'] * (3.141592653589793 / 180)
m3[['Measurement', 'Unit', 'Latitude', 'latitude_radians']].head()

,Measurement,Unit,Latitude,latitude_radians
0,0.020000,pieces/m3,-58.428300,-1.019766
1,0.008000,pieces/m3,-51.308200,-0.895497
2,0.019886,pieces/m3,-51.826667,-0.904546
3,0.018000,pieces/m3,-31.696000,-0.553200
4,0.000000,pieces/m3,6.350000,0.110828


## A column and a column

In [12]:
url = 'https://eds-217-essential-python.github.io/data/banana_index.csv'
foods = pd.read_csv(url, index_col='entity')
foods[['emissions_kg', 'land_use_kg', 'Bananas index (kg)']].head()

,emissions_kg,land_use_kg,Bananas index (kg)
entity,,,
Ale,0.488690,0.811485,0.559558
Almond butter,0.387011,7.683045,0.443134
Almond milk,0.655888,1.370106,0.751002
Almonds,0.602368,8.230927,0.689721
Apple juice,0.458378,0.660629,0.524851


In [13]:
banana_emissions = foods.loc['Bananas', 'emissions_kg']
banana_emissions

0.87334957

In [14]:
# making a new column

foods['my_banana_index'] = foods['emissions_kg'] / banana_emissions
foods[['emissions_kg', 'Bananas index (kg)', 'my_banana_index']].head()

,emissions_kg,Bananas index (kg),my_banana_index
entity,,,
Ale,0.488690,0.559558,0.559558
Almond butter,0.387011,0.443134,0.443134
Almond milk,0.655888,0.751002,0.751002
Almonds,0.602368,0.689721,0.689721
Apple juice,0.458378,0.524851,0.524851


In [15]:
foods['land_per_emission'] = foods['land_use_kg'] / foods['emissions_kg']
foods['land_per_emission'].sort_values(ascending=False).head(5)

entity
Almond butter    19.852250
Almonds          13.664286
Beans            12.428466
Chickpeas        11.578514
Lentils          10.831714
Name: land_per_emission, dtype: float64

In [ ]:
# what to do when a column spans orders of magnitude?
positive = m3[m3['Measurement'] > 0].copy()
positive.shape

(7091, 24)

In [19]:
positive['log10_measurement'] = np.log10(positive['Measurement'])
positive[['Measurement', 'log10_measurement']].head()

,Measurement,log10_measurement
0,0.020000,-1.698970
1,0.008000,-2.096910
2,0.019886,-1.701453
3,0.018000,-1.744727
5,0.013000,-1.886057


In [20]:
positive['log10_measurement'].describe()

count    7091.000000
mean       -1.254441
std         1.502175
min        -3.170053
25%        -2.188425
50%        -1.665546
75%        -0.879686
max         5.043284
Name: log10_measurement, dtype: float64

In [21]:
# Add a column to positive called log10_per_liter holding 
#   the base-10 logarithm of the pieces_per_liter column 
positive['log10_per_liter'] = np.log10(positive['pieces_per_liter'])
positive[['pieces_per_liter', 'log10_per_liter']].head()

,pieces_per_liter,log10_per_liter
0,0.000020,-4.698970
1,0.000008,-5.096910
2,0.000020,-4.701453
3,0.000018,-4.744727
5,0.000013,-4.886057


### Tidying text with .str



In [ ]:
# .str.strip() removes stray whitespace
# an empty result is an answer, and that it is usually a spelling problem
# (i.e. extra whitespace)

plastics[plastics['Keywords'] == 'R/V Tara'].shape

(0, 22)

In [23]:
plastics['Keywords'].unique()[10:14]

array(['Amazon Continental Shelf', 'Antarctic Circumnavigation Expedition', 'R/V Tara ',
       'SV Mir; ORV Alguita; SV Sea Dragon; RV Stad Amsterdam'], dtype=object)

In [ ]:
plastics[plastics['Keywords'] == 'R/V Tara '].shape
# this shows that there is some unneccessary trailing whitespace after "R/V Tara"
# "R/V Tara" vs. "R/V Tara  "

(23, 22)

In [25]:
# use .str.strip() to remove!

plastics['Keywords'] = plastics['Keywords'].str.strip()
plastics[plastics['Keywords'] == 'R/V Tara'].shape

(23, 22)

### .str.lower() makes matching predictable


In [26]:
plastics['Density Class'].value_counts()

Density Class
Medium       8029
Very Low     4155
Low          1944
High         1671
Very High     446
Name: count, dtype: int64

In [27]:
plastics['density_class'] = plastics['Density Class'].str.lower()
plastics['density_class'].value_counts()

density_class
medium       8029
very low     4155
low          1944
high         1671
very high     446
Name: count, dtype: int64

### .str.replace() swaps one piece of text for another


In [28]:
plastics['Oceans'].value_counts()

Oceans
Atlantic Ocean    14483
Pacific Ocean      1402
Arctic Ocean         69
Southern Ocean       20
Name: count, dtype: int64

In [29]:
plastics['ocean'] = plastics['Oceans'].str.replace(' Ocean', '')
plastics['ocean'].value_counts()

ocean
Atlantic    14483
Pacific      1402
Arctic         69
Southern       20
Name: count, dtype: int64